In [2]:
from langchain_community.document_loaders import CSVLoader
loader = CSVLoader(
    file_path = "books.csv",
    encoding = "utf-8",
    source_column = "url",
    content_columns = ["text"],
    metadata_columns = ["title","category","price","rating","availability"]
)

docs = loader.load()

In [3]:
docs[:3]

[Document(metadata={'source': 'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html', 'row': 0, 'title': 'A Light in the Attic', 'category': 'Poetry', 'price': '51.77', 'rating': '3', 'availability': '22'}, page_content="text: Title: A Light in the Attic\nCategory: Poetry\nPrice: £51.77\nRating: 3/5\nAvailability: 22 copies available\nDescription: It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdie

In [4]:
len(docs)

1000

In [5]:
# Skipped chunking and put same docs as chunks
# Because every doc have a meaning full data and having max 1000 characters in it and more chunks may cause the more confusion.
chunks = docs

In [6]:
print(len(chunks))

1000


In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

d:\FAHAD\New folder\Scubel\Ai Engineer\RAG1\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1842.60it/s]


In [8]:
vectors = embeddings.embed_documents([chunk.page_content for chunk in chunks])

In [9]:
len(vectors)

1000

In [10]:
from dotenv import load_dotenv
load_dotenv()
persist_direct = "books_db"

store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_direct,
    collection_name= "BOOKS"
)

retreivers = store.as_retriever(search_kwargs={"k":5})


In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq


In [12]:
llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0.2)

In [19]:
system = """Your work is to select the books as per the user's requirements with the price, rating, availability, and summarize the description concisely.
If the question asked is not related, politely reply as: "Not a related query, please try again."
Context:
{context}"""

In [20]:
prompt = ChatPromptTemplate.from_messages([
    ("system",system),
    ("human","{input}"),
])

In [21]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
document_stuff_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)

In [22]:
# Create retriver chain
from langchain_classic.chains import create_retrieval_chain
chain = create_retrieval_chain(retreivers, document_stuff_chain )

In [23]:
from groq import Groq

client = Groq()   # reads GROQ_API_KEY automatically
for m in client.models.list().data:
    print(m.id)

openai/gpt-oss-20b
whisper-large-v3
canopylabs/orpheus-v1-english
meta-llama/llama-prompt-guard-2-86m
openai/gpt-oss-safeguard-20b
qwen/qwen3.8-27b
openai/gpt-oss-120b
whisper-large-v3-turbo
meta-llama/llama-prompt-guard-2-22m
allam-2-7b
canopylabs/orpheus-arabic-saudi


In [24]:
response = chain.invoke({"input" : "Give a book which demonstrate the love."})
response

{'input': 'Give a book which demonstrate the love.',
 'context': [Document(metadata={'row': 891, 'rating': '3', 'category': 'Fiction', 'availability': '2', 'title': 'The Course of Love', 'source': 'https://books.toscrape.com/catalogue/the-course-of-love_109/index.html', 'price': '16.78'}, page_content='text: Title: The Course of Love\nCategory: Fiction\nPrice: £16.78\nRating: 3/5\nAvailability: 2 copies available\nDescription: The long-awaited and beguiling second novel from Alain de Botton that tracks the beautifully complicated arc of a romantic partnership, from the internationally bestselling author of On Love and How Proust Can Change Your Life.We all know the headiness and excitement of the early days of love. But what comes after? In Edinburgh, a couple, Rabih and Kirsten, fall in love. T The long-awaited and beguiling second novel from Alain de Botton that tracks the beautifully complicated arc of a romantic partnership, from the internationally bestselling author of On Love an

In [25]:
response = chain.invoke({"input" : "How many books cost less than £10?"})
response

{'input': 'How many books cost less than £10?',
 'context': [Document(metadata={'title': 'The Little Paris Bookshop', 'row': 593, 'source': 'https://books.toscrape.com/catalogue/the-little-paris-bookshop_407/index.html', 'availability': '5', 'rating': '3', 'category': 'Fiction', 'price': '24.73'}, page_content="text: Title: The Little Paris Bookshop\nCategory: Fiction\nPrice: £24.73\nRating: 3/5\nAvailability: 5 copies available\nDescription: “There are books that are suitable for a million people, others for only a hundred. There are even remedies—I mean books—that were written for one person only…A book is both medic and medicine at once. It makes a diagnosis as well as offering therapy. Putting the right novels to the appropriate ailments: that’s how I sell books.”Monsieur Perdu calls himself a literary apot “There are books that are suitable for a million people, others for only a hundred. There are even remedies—I mean books—that were written for one person only…A book is both med